# Hamilton Prep: tip pick-up and drop

Exercises the tip pick-up and drop paths the Prep driver has, checking each one three ways:

1. **the device** - what the channels' sleeve sensors report, which is a measurement, not a model;
2. **the model** - which channel carries a tip and which spots hold one, in the resource tree;
3. **the wire** - the commands each step sent, counted as it runs and listed at the end.

`SIMULATION = True` runs it with no instrument. Set it to `False`, fill in `HOST`, and run the same
cells on the device.

**Before running on the instrument**

- The rack on `TIP_RACK_SPOT` holds tips in the working column, and **its spare column is empty** -
  take those tips out by hand, since step 4 drops tips there.
- Every step prints what it is about to do before it moves, and each check prints sensed against
  modelled. If the two disagree, stop: each step assumes the channels are as the last one left them.
- Step 7 is the only one where nothing should move.

In [1]:
SIMULATION = True
HOST = None  # e.g. "192.168.1.50", when SIMULATION is False

TIP_RACK_SPOT = 3  # the deck spot the tip rack stands on
COLUMN = 1  # the rack column to pick up from; holds tips
SPARE_COLUMN = 12  # a column that is EMPTY on the bench; step 4 drops tips into it

In [2]:
import logging
import sys

from pylabrobot.hamilton.prep import Prep
from pylabrobot.resources import Coordinate, set_tip_tracking
from pylabrobot.resources.errors import HasTipError, NoTipError
from pylabrobot.resources.hamilton import PrepDeck, hamilton_96_tiprack_50uL_NTR

logging.getLogger("pylabrobot").setLevel(logging.INFO)
logging.getLogger("pylabrobot").handlers.clear()
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(levelname)s - %(message)s"))
logging.getLogger("pylabrobot").addHandler(handler)

set_tip_tracking(True)  # a spot gives up its tip, and takes it back

## Deck and device

In [3]:
deck = PrepDeck()
tip_rack = deck[TIP_RACK_SPOT] = hamilton_96_tiprack_50uL_NTR(name="tips", with_tips=True)

prep = Prep(deck=deck, simulation=SIMULATION, host=HOST)
await prep.setup()

channels = prep.driver.pipettes
head8 = prep.driver.head8
n = channels.num_channels
rows = "ABCDEFGH"[:n]  # the rows this many channels reach at the rack's pitch
print(f"channels: {n}, 8-channel head: {head8 is not None}")

# The spare column is empty on the bench, so the model has to say so too.
for row in "ABCDEFGH":
  spot = tip_rack.get_item(f"{row}{SPARE_COLUMN}")
  if spot.has_tip():
    spot.unassign_tip()
print(f"spare column {SPARE_COLUMN} emptied in the model; it must be empty on the bench too")

INFO - tips held at setup: none


INFO - [Hamilton Prep] Connected on simulation (no link)
  Serial: PRPAA1087
  Firmware: MLPrep Runtime V1.2.2.444 99020-02 Rev G
  Configuration: enclosure none, safe speeds off, traverse height 167.5 mm
  Deck: x 0.00 to 299.00 mm, y -9.00 to 385.00 mm, z 18.03 to 167.50 mm; 3 sites, 3 waste sites
  Pipettes: 2, v2 aspirate/dispense
    channel 0 (rear): firmware Channel1ml Runtime V1.4.9.249 98228-02 Rev K, x 0.19 to 299.19 mm, y 0.00 to 385.00 mm, z 18.03 to 167.50 mm
    channel 1 (front): firmware Channel1ml Runtime V1.4.9.249 98228-02 Rev K, x 0.19 to 299.19 mm, y -9.00 to 376.00 mm, z 18.03 to 167.50 mm
  8-channel head: none


channels: 2, 8-channel head: False
spare column 12 emptied in the model; it must be empty on the bench too


## What the checks do

`check` prints what the device senses, what the model holds and what the step expected, and records
the result instead of raising, so a disagreement stops the run where it happened. `sent` counts the
commands a step put on the wire.

In [4]:
results = []
commands = []

_send = prep.driver.send_command


async def recording(command, **kwargs):
  commands.append(type(command).__name__)
  return await _send(command, **kwargs)


prep.driver.send_command = recording  # type: ignore[method-assign]


def held(column):
  """Which spots of a column hold a tip, in the model."""
  return [f"{row}{column}" for row in "ABCDEFGH" if tip_rack.get_item(f"{row}{column}").has_tip()]


async def check(step, carrying, column=None, expect_held=None):
  """Compare the device, the model and what the step expected."""
  device = [bool(x) for x in await channels.sense_tip_presence()]
  model = [channels.shaft(ch).has_tip() for ch in range(n)]
  expected = [ch in carrying for ch in range(n)]
  ok = device == expected and model == expected
  print(f"   sensed   {device}")
  print(f"   modelled {model}")
  print(f"   expected {expected}")
  if column is not None:
    spots = held(column)
    ok = ok and spots == expect_held
    print(f"   column {column} holds {spots or 'nothing'}, expected {expect_held or 'nothing'}")
  print(f"   {'OK' if ok else 'MISMATCH - stop and look'}: {step}")
  results.append((step, ok))


def sent(step):
  """Print and clear the commands this step sent."""
  counts = {name: commands.count(name) for name in dict.fromkeys(commands)}
  print(f"   sent: {counts or 'nothing'}")
  commands.clear()

## 1. One channel

The smallest move there is: channel 0 takes the tip from the first row and puts it back.

In [5]:
first = tip_rack.get_item(f"A{COLUMN}")
full = [f"{row}{COLUMN}" for row in "ABCDEFGH"]

print(f"channel 0 picks up from {first.name}")
commands.clear()
await channels.pick_up_tips([first], use_channels=[0])
sent("pick up")
await check("1a. channel 0 picked up", carrying=[0], column=COLUMN, expect_held=full[1:])

print("channel 0 drops it back")
await channels.drop_tips([first], use_channels=[0])
sent("drop")
await check("1b. channel 0 dropped it back", carrying=[], column=COLUMN, expect_held=full)

channel 0 picks up from tips_tipspot_A1
   sent: {'PrepGetPositions': 2, 'PrepGetXSpeedScale': 1, 'PrepSetXSpeedScale': 2, 'PrepMoveToPosition': 1, 'PrepPickUpTips': 1}
   sensed   [True, False]
   modelled [True, False]
   expected [True, False]
   column 1 holds ['B1', 'C1', 'D1', 'E1', 'F1', 'G1', 'H1'], expected ['B1', 'C1', 'D1', 'E1', 'F1', 'G1', 'H1']
   OK: 1a. channel 0 picked up
channel 0 drops it back
   sent: {'PrepProbeRequest': 2, 'PrepDropTips': 1}
   sensed   [False, False]
   modelled [False, False]
   expected [False, False]
   column 1 holds ['A1', 'B1', 'C1', 'D1', 'E1', 'F1', 'G1', 'H1'], expected ['A1', 'B1', 'C1', 'D1', 'E1', 'F1', 'G1', 'H1']
   OK: 1b. channel 0 dropped it back


## 2. Two adjacent channels

Channels 0 and 1 onto the first two rows, 9 mm apart: the first move where the channels are spaced.

In [6]:
pair = tip_rack[[f"A{COLUMN}", f"B{COLUMN}"]]
commands.clear()
await channels.pick_up_tips(pair, use_channels=[0, 1])
sent("pick up")
await check("2a. channels 0 and 1 picked up", carrying=[0, 1], column=COLUMN, expect_held=full[2:])

await channels.drop_tips(pair, use_channels=[0, 1])
sent("drop")
await check("2b. channels 0 and 1 dropped back", carrying=[], column=COLUMN, expect_held=full)

   sent: {'PrepGetPositions': 2, 'PrepMoveToPosition': 1, 'PrepPickUpTips': 1}
   sensed   [True, True]
   modelled [True, True]
   expected [True, True]
   column 1 holds ['C1', 'D1', 'E1', 'F1', 'G1', 'H1'], expected ['C1', 'D1', 'E1', 'F1', 'G1', 'H1']
   OK: 2a. channels 0 and 1 picked up
   sent: {'PrepProbeRequest': 2, 'PrepDropTips': 1}
   sensed   [False, False]
   modelled [False, False]
   expected [False, False]
   column 1 holds ['A1', 'B1', 'C1', 'D1', 'E1', 'F1', 'G1', 'H1'], expected ['A1', 'B1', 'C1', 'D1', 'E1', 'F1', 'G1', 'H1']
   OK: 2b. channels 0 and 1 dropped back


## 3. Every channel at once

In [7]:
column = tip_rack[[f"{row}{COLUMN}" for row in rows]]
everyone = list(range(n))

commands.clear()
await channels.pick_up_tips(column, use_channels=everyone)
sent("pick up")
await check("3a. every channel picked up", carrying=everyone, column=COLUMN, expect_held=full[n:])

await channels.drop_tips(column, use_channels=everyone)
sent("drop")
await check("3b. every channel dropped back", carrying=[], column=COLUMN, expect_held=full)

   sent: {'PrepGetPositions': 2, 'PrepMoveToPosition': 1, 'PrepPickUpTips': 1}
   sensed   [True, True]
   modelled [True, True]
   expected [True, True]
   column 1 holds ['C1', 'D1', 'E1', 'F1', 'G1', 'H1'], expected ['C1', 'D1', 'E1', 'F1', 'G1', 'H1']
   OK: 3a. every channel picked up
   sent: {'PrepProbeRequest': 2, 'PrepDropTips': 1}
   sensed   [False, False]
   modelled [False, False]
   expected [False, False]
   column 1 holds ['A1', 'B1', 'C1', 'D1', 'E1', 'F1', 'G1', 'H1'], expected ['A1', 'B1', 'C1', 'D1', 'E1', 'F1', 'G1', 'H1']
   OK: 3b. every channel dropped back


## 4. Dropping somewhere else

Tips come off the working column and go into the spare one, so the drop is not the pick-up run
backwards. They are carried back afterwards, leaving the rack as it started.

In [8]:
spare = tip_rack[[f"{row}{SPARE_COLUMN}" for row in rows]]
spare_names = [f"{row}{SPARE_COLUMN}" for row in rows]

commands.clear()
await channels.pick_up_tips(column, use_channels=everyone)
await check("4a. picked up from the working column", carrying=everyone)

await channels.drop_tips(spare, use_channels=everyone)
sent("pick up and drop")
await check("4b. dropped into the spare column", carrying=[], column=SPARE_COLUMN, expect_held=spare_names)

await channels.pick_up_tips(spare, use_channels=everyone)
await channels.drop_tips(column, use_channels=everyone)
await check("4c. carried back to the working column", carrying=[], column=COLUMN, expect_held=full)

   sensed   [True, True]
   modelled [True, True]
   expected [True, True]
   OK: 4a. picked up from the working column
   sent: {'PrepGetPositions': 2, 'PrepMoveToPosition': 1, 'PrepPickUpTips': 1, 'PrepProbeRequest': 2, 'PrepDropTips': 1}
   sensed   [False, False]
   modelled [False, False]
   expected [False, False]
   column 12 holds ['A12', 'B12'], expected ['A12', 'B12']
   OK: 4b. dropped into the spare column
   sensed   [False, False]
   modelled [False, False]
   expected [False, False]
   column 1 holds ['A1', 'B1', 'C1', 'D1', 'E1', 'F1', 'G1', 'H1'], expected ['A1', 'B1', 'C1', 'D1', 'E1', 'F1', 'G1', 'H1']
   OK: 4c. carried back to the working column


## 5. An offset

The same pick-up, 1 mm right and 1 mm back. On the instrument, watch where the channel descends.

In [9]:
commands.clear()
await channels.pick_up_tips([first], use_channels=[0], offsets=[Coordinate(1.0, 1.0, 0.0)])
sent("pick up with offset")
await check("5a. picked up with a 1 mm offset", carrying=[0])

await channels.drop_tips([first], use_channels=[0], offsets=[Coordinate(1.0, 1.0, 0.0)])
await check("5b. dropped with the same offset", carrying=[], column=COLUMN, expect_held=full)

   sent: {'PrepGetPositions': 2, 'PrepGetXSpeedScale': 1, 'PrepSetXSpeedScale': 2, 'PrepMoveToPosition': 1, 'PrepPickUpTips': 1}
   sensed   [True, False]
   modelled [True, False]
   expected [True, False]
   OK: 5a. picked up with a 1 mm offset
   sensed   [False, False]
   modelled [False, False]
   expected [False, False]
   column 1 holds ['A1', 'B1', 'C1', 'D1', 'E1', 'F1', 'G1', 'H1'], expected ['A1', 'B1', 'C1', 'D1', 'E1', 'F1', 'G1', 'H1']
   OK: 5b. dropped with the same offset


## 6. A rack with tips missing

The first two spots are emptied in the model - **take those two tips out by hand before running this
on the instrument** - and the channels pick up from spots further down the column.

In [10]:
missing = [tip_rack.get_item(f"{row}{COLUMN}") for row in "AB"]
for spot in missing:
  if spot.has_tip():
    spot.unassign_tip()
print(f"column {COLUMN} now holds {held(COLUMN)}")

lower_rows = "ABCDEFGH"[2 : 2 + n]
lower = tip_rack[[f"{row}{COLUMN}" for row in lower_rows]]
commands.clear()
await channels.pick_up_tips(lower, use_channels=everyone)
sent("pick up")
await check("6a. picked up from the spots that held tips", carrying=everyone)

await channels.drop_tips(lower, use_channels=everyone)
await check(
  "6b. dropped them back", carrying=[], column=COLUMN, expect_held=[f"{row}{COLUMN}" for row in "ABCDEFGH"[2:]]
)

for spot in missing:  # the two tips go back on the bench, and in the model
  spot.assign_tip(spot.make_tip())

column 1 now holds ['C1', 'D1', 'E1', 'F1', 'G1', 'H1']
   sent: {'PrepGetPositions': 2, 'PrepGetXSpeedScale': 1, 'PrepSetXSpeedScale': 2, 'PrepMoveToPosition': 1, 'PrepPickUpTips': 1}
   sensed   [True, True]
   modelled [True, True]
   expected [True, True]
   OK: 6a. picked up from the spots that held tips
   sensed   [False, False]
   modelled [False, False]
   expected [False, False]
   column 1 holds ['C1', 'D1', 'E1', 'F1', 'G1', 'H1'], expected ['C1', 'D1', 'E1', 'F1', 'G1', 'H1']
   OK: 6b. dropped them back


## 7. What the driver refuses

Nothing moves here. Each call should raise before it reaches the machine.

In [11]:
async def refuses(description, call):
  try:
    await call()
  except (NoTipError, HasTipError, RuntimeError, ValueError) as e:
    print(f"   refused, as it should be: {description} -> {type(e).__name__}: {e}")
    results.append((f"7. {description}", True))
  else:
    print(f"   NOT REFUSED: {description}")
    results.append((f"7. {description}", False))


empty = tip_rack.get_item(f"A{COLUMN}")
empty.unassign_tip()
await refuses("pick up from an empty spot", lambda: channels.pick_up_tips([empty], use_channels=[0]))
empty.assign_tip(empty.make_tip())

await channels.pick_up_tips([first], use_channels=[0])
await refuses(
  "pick up onto a channel that carries a tip",
  lambda: channels.pick_up_tips([tip_rack.get_item(f"B{COLUMN}")], use_channels=[0]),
)
await channels.drop_tips([first], use_channels=[0])

await refuses("drop from a channel with no tip", lambda: channels.drop_tips([first], use_channels=[0]))
await refuses(
  "drop into a spot that already holds a tip",
  lambda: channels.drop_tips([tip_rack.get_item(f"B{COLUMN}")], use_channels=[0]),
)

   refused, as it should be: pick up from an empty spot -> NoTipError: tips_tipspot_A1 holds no tip
   refused, as it should be: pick up onto a channel that carries a tip -> HasTipError: Channel 0 already carries tips_tipspot_A1#2
   refused, as it should be: drop from a channel with no tip -> NoTipError: No tip mounted on channel 0; call pick_up_tips first.
   refused, as it should be: drop into a spot that already holds a tip -> NoTipError: No tip mounted on channel 0; call pick_up_tips first.


## 8. Repeats

Three cycles on the same column. A seating problem shows here rather than on the first try: watch for
a channel that senses no tip after a pick-up, or keeps one after a drop.

In [12]:
for cycle in range(1, 4):
  commands.clear()
  await channels.pick_up_tips(column, use_channels=everyone)
  await check(f"8.{cycle}a. cycle {cycle} picked up", carrying=everyone)
  await channels.drop_tips(column, use_channels=everyone)
  await check(f"8.{cycle}b. cycle {cycle} dropped", carrying=[], column=COLUMN, expect_held=full)
  sent(f"cycle {cycle}")

   sensed   [True, True]
   modelled [True, True]
   expected [True, True]
   OK: 8.1a. cycle 1 picked up
   sensed   [False, False]
   modelled [False, False]
   expected [False, False]
   column 1 holds ['A1', 'B1', 'C1', 'D1', 'E1', 'F1', 'G1', 'H1'], expected ['A1', 'B1', 'C1', 'D1', 'E1', 'F1', 'G1', 'H1']
   OK: 8.1b. cycle 1 dropped
   sent: {'PrepGetPositions': 2, 'PrepMoveToPosition': 1, 'PrepPickUpTips': 1, 'PrepProbeRequest': 4, 'PrepDropTips': 1}
   sensed   [True, True]
   modelled [True, True]
   expected [True, True]
   OK: 8.2a. cycle 2 picked up
   sensed   [False, False]
   modelled [False, False]
   expected [False, False]
   column 1 holds ['A1', 'B1', 'C1', 'D1', 'E1', 'F1', 'G1', 'H1'], expected ['A1', 'B1', 'C1', 'D1', 'E1', 'F1', 'G1', 'H1']
   OK: 8.2b. cycle 2 dropped
   sent: {'PrepGetPositions': 2, 'PrepMoveToPosition': 1, 'PrepPickUpTips': 1, 'PrepProbeRequest': 4, 'PrepDropTips': 1}
   sensed   [True, True]
   modelled [True, True]
   expected [True, True]

## 9. The 8-channel head

Skipped on a Prep without one. The head takes a whole column at once, and its own shafts carry the
tips.

In [13]:
if head8 is None:
  print("no 8-channel head on this device; skipped")
else:
  head_column = tip_rack[[f"{row}{COLUMN}" for row in "ABCDEFGH"]]
  commands.clear()
  await head8.pick_up_tips(head_column)
  carried = [head8.shaft(ch).has_tip() for ch in range(8)]
  print(f"   head shafts carrying tips: {carried}")
  print(f"   column {COLUMN} holds {held(COLUMN) or 'nothing'}")
  results.append(("9a. head picked up a column", all(carried) and held(COLUMN) == []))
  sent("head pick up")

  await head8.drop_tips(head_column)
  carried = [head8.shaft(ch).has_tip() for ch in range(8)]
  print(f"   head shafts after the drop: {carried}")
  print(f"   column {COLUMN} holds {held(COLUMN)}")
  results.append(("9b. head dropped the column back", not any(carried) and held(COLUMN) == full))
  sent("head drop")

no 8-channel head on this device; skipped


## Summary

In [14]:
width = max(len(step) for step, _ in results)
for step, ok in results:
  print(f"{step:{width}}  {'OK' if ok else 'MISMATCH'}")
failed = [step for step, ok in results if not ok]
print()
print(f"{len(results) - len(failed)} of {len(results)} checks as expected")
if failed:
  print("look at: " + ", ".join(failed))

1a. channel 0 picked up                       OK
1b. channel 0 dropped it back                 OK
2a. channels 0 and 1 picked up                OK
2b. channels 0 and 1 dropped back             OK
3a. every channel picked up                   OK
3b. every channel dropped back                OK
4a. picked up from the working column         OK
4b. dropped into the spare column             OK
4c. carried back to the working column        OK
5a. picked up with a 1 mm offset              OK
5b. dropped with the same offset              OK
6a. picked up from the spots that held tips   OK
6b. dropped them back                         OK
7. pick up from an empty spot                 OK
7. pick up onto a channel that carries a tip  OK
7. drop from a channel with no tip            OK
7. drop into a spot that already holds a tip  OK
8.1a. cycle 1 picked up                       OK
8.1b. cycle 1 dropped                         OK
8.2a. cycle 2 picked up                       OK
8.2b. cycle 2 droppe

In [15]:
await prep.stop()

INFO - Hamilton TCP client stopped
